In [ ]:
# Install dependencies
!pip uninstall -y -q torch torchvision torchaudio cudf easyocr numba-cuda dask-cuda pylibraft-cu12 raft-dask-cu12 pylibcudf-cu12 rmm-cu12 quantecon cuvs-cu12 ucxx-cu12 cuml-cu12 cuda-python cuml libcuml libcuvs libraft numba ydata-profiling google-colab
!pip install -q torch==2.5.1+cu121 torchvision==0.20.1+cu121 --index-url https://download.pytorch.org/whl/cu121
!pip install -q torch>=2.1.0 transformers>=5.15.0 datasets>=2.14.0 numpy>=1.24.0 matplotlib>=3.7.0 scipy>=1.10.0
!pip install -q -U transformers


In [ ]:
import os
import sys
import time
import traceback
import torch
# KAGGLEHUB_CACHE: persist models across sessions
os.environ["KAGGLEHUB_CACHE"] = "/kaggle/working/.model_cache"
os.makedirs("/kaggle/working/.model_cache", exist_ok=True)
WORKING = "/kaggle/working"
sys.path.insert(0, WORKING)
num_gpus = torch.cuda.device_count()
if num_gpus >= 2:
    device_L, device_S = "cuda:0", "cuda:1"
    dtype_L = dtype_S = torch.float32
elif num_gpus == 1:
    device_L = device_S = "cuda:0"
    dtype_L = torch.float16
    dtype_S = torch.float32
else:
    device_L = device_S = "cpu"
    dtype_L = dtype_S = torch.float32
if num_gpus > 0:
    print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"torch {torch.__version__} | L={device_L}, S={device_S}")
RES = os.path.join(WORKING, "results")
os.makedirs(RES, exist_ok=True)
def resolve(local_dir, hf_id):
    return local_dir if os.path.isdir(local_dir) else hf_id
SMALL = resolve("/kaggle/input/qwen35-08b", "Qwen/Qwen3.5-0.8B")
LARGE = resolve("/kaggle/input/qwen35-4b", "Qwen/Qwen3.5-4B")


In [ ]:
import torch
from transformers import DynamicCache
from transformers.cache_utils import (  # type: ignore[import-not-found]
    DynamicLayer,
    LinearAttentionLayer,
)
def compute_jacobian_for_tokens(
    model,
    layer_idx: int,
    input_ids: torch.Tensor,
    attention_mask: torch.Tensor,
    token_set: list[int],
) -> torch.Tensor:
    """Compute Jacobian of logits w.r.t. hidden state at layer_idx.
    Args:
        model: Qwen3_5ForCausalLM instance.
        layer_idx: Layer index (0-based) to compute Jacobian at.
        input_ids: [1, S] token IDs.
        attention_mask: [1, S] attention mask.
        token_set: list of token IDs to compute gradients for.
    Returns:
        J: [|U|, D] Jacobian matrix (unnormalized).
    """
    model.eval()
    S = input_ids.shape[1]
    prefix_cache = _run_prefix_pass(model, input_ids, attention_mask)
    h = _extract_hidden_state(model, input_ids, attention_mask, layer_idx)
    # Match model dtype to avoid mismatch in decoder (fp16 models on GPU)
    model_dtype = next(model.parameters()).dtype
    h = h.to(model_dtype).detach().requires_grad_(True)
    logits = _run_decoder_pass(model, prefix_cache, h, S, layer_idx)
    J = _compute_jacobian_rows(logits, h, token_set)
    return J
def _run_prefix_pass(
    model, input_ids: torch.Tensor, attention_mask: torch.Tensor
) -> DynamicCache:
    """Prefix pass: all tokens except last -> cache."""
    with torch.no_grad():
        outputs = model(
            input_ids=input_ids[:, :-1],
            attention_mask=attention_mask[:, :-1],
            use_cache=True,
            output_hidden_states=False,
        )
    return outputs.past_key_values
def _extract_hidden_state(model, input_ids, attention_mask, layer_idx) -> torch.Tensor:
    """Full pass: extract hidden state at layer_idx for last token."""
    with torch.no_grad():
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            output_hidden_states=True,
            use_cache=False,
        )
    # hidden_states[0] = embeddings, [l+1] = output of layer l
    h = outputs.hidden_states[layer_idx + 1][:, -1:, :]
    return h  # [1, 1, D]
def _deep_clone_cache(cache):
    """Deep clone cache, ensuring all tensors are independent copies.
    ponytail: copy.deepcopy works but is slow. Manual clone for speed + safety.
    """
    new_cache = DynamicCache()
    for idx in range(len(cache.layers)):
        layer = cache.layers[idx]
        if isinstance(layer, DynamicLayer):
            new_k = [k.clone() if k is not None else None for k in layer.keys]
            new_v = [v.clone() if v is not None else None for v in layer.values]
            new_cache.layers.append(DynamicLayer(keys=new_k, values=new_v))
        else:
            new_conv = {k: v.clone() for k, v in layer.conv_states.items()}
            new_recur = {k: v.clone() for k, v in layer.recurrent_states.items()}
            new_cache.layers.append(
                LinearAttentionLayer(conv_states=new_conv, recurrent_states=new_recur)
            )
    return new_cache
def _run_decoder_pass(
    model, prefix_cache, h: torch.Tensor, S: int, layer_idx: int
) -> torch.Tensor:
    """Decode h through remaining layers using prefix cache."""
    # Clone cache tensors independently to avoid in-place mutation
    cache_clone = _deep_clone_cache(prefix_cache)
    text_model = model.model  # Qwen3_5TextModel
    pos = torch.full((4, 1, 1), S - 1, dtype=torch.long, device=h.device)
    text_position_ids = pos[0]
    rope_position_ids = pos[1:]
    position_embeddings = text_model.rotary_emb(h, rope_position_ids)
    x = h
    n_layers = model.config.num_hidden_layers
    start = layer_idx + 1
    causal_mask_mapping = _build_causal_masks(
        text_model, text_position_ids, cache_clone, x
    )
    for i in range(start, n_layers):
        layer_type = model.config.layer_types[i]
        attn_mask = causal_mask_mapping[layer_type]
        x = text_model.layers[i](
            x,
            position_embeddings=position_embeddings,
            attention_mask=attn_mask,
            position_ids=text_position_ids,
            past_key_values=cache_clone,
        )
    x = text_model.norm(x)
    logits = model.lm_head(x[:, -1, :])
    return logits
def _build_causal_masks(text_model, text_pos, cache, x):
    """Build causal mask mapping for a single token with cache."""
    from transformers.masking_utils import (
        create_causal_mask,
        create_recurrent_attention_mask,
    )
    dummy_embeds = x.new_zeros(1, 1, text_model.config.hidden_size)
    mask_kwargs = {
        "config": text_model.config,
        "inputs_embeds": dummy_embeds,
        "attention_mask": None,
        "past_key_values": cache,
        "position_ids": text_pos,
    }
    return {
        "full_attention": create_causal_mask(**mask_kwargs),
        "linear_attention": create_recurrent_attention_mask(**mask_kwargs),
    }
def _compute_jacobian_rows(
    logits: torch.Tensor, h: torch.Tensor, token_set: list[int]
) -> torch.Tensor:
    """Compute Jacobian rows via loop over token_set."""
    logits_subset = logits[0, token_set]  # [|U|]
    D = h.shape[-1]
    J = torch.zeros(len(token_set), D, device=h.device, dtype=h.dtype)
    for i in range(len(token_set)):
        grad = torch.autograd.grad(
            outputs=logits_subset[i],
            inputs=h,
            retain_graph=(i < len(token_set) - 1),
        )[0]
        J[i] = grad[0, 0, :]
    return J
def softmax_jacobian(J: torch.Tensor, p: torch.Tensor) -> torch.Tensor:
    """Gauge-invariant Jacobian: J^(p) = diag(p) - pp^T @ J.
    J^(p)_i = p_i * (J_i - J_bar_p),  J_bar_p = sum_k p_k * J_k
    Removes the common-mode shift (gauge) from raw Jacobian.
    """
    J_bar = (p[:, None] * J).sum(dim=0, keepdim=True)  # [1, D]
    return p[:, None] * (J - J_bar)  # [K, D]
def uniform_centered(J: torch.Tensor) -> torch.Tensor:
    """Uniform centering: J - mean(J). Removes average shift."""
    return J - J.mean(dim=0, keepdim=True)


In [ ]:
import hashlib
import os
import torch
_gramian_cache: dict[str, torch.Tensor] = {}
_GRAMIAN_DIR = os.path.join("results", ".gramians")
def jacobian_bridge(
    J_L: torch.Tensor,
    J_S: torch.Tensor,
    E_L: torch.Tensor,
    E_S: torch.Tensor,
    probs_L: torch.Tensor,
    threshold: float = 1e-6,
) -> float:
    """Compute Jacobian bridge loss between large and small models.
    The bridge chain: J_L @ E_L^T @ E_S @ (E_S^T @ E_S)^{-1}.
    The inverse Gramian (E_S^T @ E_S)^{-1} corrects for non-orthogonal
    embeddings, so identity (J_L == J_S, E_L == E_S) yields loss ~0.
    """
    G_inv = _inverse_gramian(E_S)
    J_L_proj = _project_to_vocab(J_L, E_L)
    J_S_target = _project_to_small(J_L_proj, E_S, G_inv)
    mask = _valid_mask(J_S, J_S_target, threshold)
    if mask.sum() == 0:
        return 1.0
    return _weighted_cosine_loss(J_S, J_S_target, probs_L, mask)
def _project_to_vocab(J_L: torch.Tensor, E_L: torch.Tensor) -> torch.Tensor:
    """Project J_L [|U|, D_L] through E_L^T -> [|U|, V]."""
    return J_L.float() @ E_L.float().T
def _project_to_small(
    J_L_proj: torch.Tensor, E_S: torch.Tensor, G_inv: torch.Tensor
) -> torch.Tensor:
    """Project [|U|, V] through E_S @ G_inv -> [|U|, D_S].
    G_inv = (E_S^T @ E_S)^{-1} corrects non-orthogonal embeddings.
    """
    return J_L_proj @ E_S.float() @ G_inv
def _inverse_gramian(E: torch.Tensor) -> torch.Tensor:
    """Compute (E^T @ E)^{-1}, cached on disk per embedding hash."""
    key = _gramian_key(E)
    if key not in _gramian_cache:
        path = os.path.join(_GRAMIAN_DIR, f"{key}.pt")
        if os.path.exists(path):
            _gramian_cache[key] = torch.load(path, weights_only=True)
        else:
            _gramian_cache[key] = _compute_inverse_gramian(E)
            os.makedirs(_GRAMIAN_DIR, exist_ok=True)
            torch.save(_gramian_cache[key], path)
    return _gramian_cache[key].to(E.device)
def _gramian_key(E: torch.Tensor) -> str:
    """Stable hash of an embedding matrix for disk caching.
    Uses first/last rows as representative content — hashing the full
    248k×1024 matrix would be prohibitively expensive.
    """
    header = f"{tuple(E.shape)}|{str(E.device)}".encode()
    data = header + E[0].cpu().numpy().tobytes() + E[-1].cpu().numpy().tobytes()
    h = hashlib.sha256(data).hexdigest()[:16]
    return f"g_{h}"
def _compute_inverse_gramian(E: torch.Tensor) -> torch.Tensor:
    """Compute (E^T @ E)^{-1} via pseudoinverse for numerical stability."""
    G = E.float().T @ E.float()
    return torch.linalg.pinv(G)
def _valid_mask(
    J_S: torch.Tensor, J_target: torch.Tensor, threshold: float
) -> torch.Tensor:
    """Mask rows where either norm < threshold."""
    n1 = J_S.float().norm(dim=1)
    n2 = J_target.float().norm(dim=1)
    return (n1 > threshold) & (n2 > threshold)
def _weighted_cosine_loss(
    J_S: torch.Tensor,
    J_target: torch.Tensor,
    probs: torch.Tensor,
    mask: torch.Tensor,
) -> float:
    """Weighted cosine dissimilarity loss."""
    cos_sim = torch.nn.functional.cosine_similarity(
        J_S.float()[mask], J_target[mask], dim=1
    )
    w = probs[mask]
    w = w / (w.sum() + 1e-8)
    return (w * (1.0 - cos_sim)).sum().item()


In [ ]:
import torch
def compute_esg(
    jacobians: list[torch.Tensor],
    n_components: int = 32,
) -> tuple[torch.Tensor, torch.Tensor]:
    """Compute ESG from a list of Jacobian matrices.
    Args:
        jacobians: list of N Jacobian matrices, each [|U|, D].
        n_components: number of top eigenvalues/vectors to return.
    Returns:
        eigenvalues: [n_components] sorted descending.
        eigenvectors: [n_components, |U|] corresponding eigenvectors.
    """
    G = _build_gramian(jacobians)
    eigenvalues, eigenvectors = _spectral_decomposition(G, n_components)
    return eigenvalues, eigenvectors
def _build_gramian(jacobians: list[torch.Tensor]) -> torch.Tensor:
    """G = (1/N) * sum_i (J_i @ J_i^T)."""
    U = jacobians[0].shape[0]
    device = jacobians[0].device
    G = torch.zeros(U, U, device=device, dtype=torch.float)
    for J in jacobians:
        G += J.float() @ J.float().T
    G /= len(jacobians)
    return G
def _spectral_decomposition(
    G: torch.Tensor, n_components: int
) -> tuple[torch.Tensor, torch.Tensor]:
    """Eigen decomposition, sorted descending."""
    eigenvalues, eigenvectors = torch.linalg.eigh(G)
    # Sort descending
    idx = eigenvalues.argsort(descending=True)
    eigenvalues = eigenvalues[idx][:n_components]
    eigenvectors = eigenvectors[:, idx][:, :n_components].T
    return eigenvalues, eigenvectors


In [ ]:
from collections import Counter
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
def uniform_layer_mapping(n_layers_large: int, n_layers_small: int) -> dict[int, int]:
    """Uniform layer mapping from large to small model layers."""
    mapping = {}
    for i in range(n_layers_large):
        j = round(i * (n_layers_small - 1) / (n_layers_large - 1))
        mapping[i] = j
    return mapping
def get_layer_types(model: AutoModelForCausalLM) -> list[str]:
    """Get layer types from model config (full_attention / linear_attention)."""
    return list(model.config.layer_types)  # type: ignore[attr-defined]
_DOMAIN_DATASETS = {
    "wiki": ("allenai/c4", "en", "train", "text"),
    "math": ("openai/gsm8k", "main", "train", "question"),
    "code": ("neyralabs/apex-agents", None, "train", "prompt"),
    "chat": ("HuggingFaceH4/ultrachat_200k", None, "train_sft", "messages"),
}
def _load_dataset_for_domain(
    domain: str,
) -> tuple[str, str | None, str, str]:
    """Return (dataset_id, config, split, text_field) for a domain.
    Falls back to wiki (C4) if the target dataset is unavailable.
    """
    ds_id, cfg, split, field = _DOMAIN_DATASETS[domain]
    try:
        kw: dict[str, object] = {"split": split, "streaming": True}
        if cfg is not None:
            kw["name"] = cfg
        load_dataset(ds_id, **kw)  # type: ignore[call-overload]
        return ds_id, cfg, split, field
    except Exception:
        print(f"  Warning: domain {domain} ({ds_id}) unavailable, falling back to C4")
        return _DOMAIN_DATASETS["wiki"]
def _extract_text_from_sample(
    sample: dict,
    field: str,
) -> str:
    """Extract text from a dataset sample, handling different formats."""
    if field == "messages":
        # Chat format: list of {role, content}
        texts = [m.get("content", "") for m in sample.get("messages", [])]
        return " ".join(texts)
    val = sample.get(field, "")
    if isinstance(val, list):
        # e.g. code_contests solutions: list of strings
        return "\n".join(str(v) for v in val)
    if isinstance(val, dict):
        return str(val)
    return val
def load_calibration_data(
    model_name: str,
    max_length: int = 512,
    min_length: int = 256,
    n_samples: int = 4,
    domain: str = "wiki",
) -> list[dict[str, torch.Tensor]]:
    """Load and tokenize calibration data.
    domain: 'wiki' (C4), 'math' (GSM8K), 'code' (apex-agents), 'chat' (ultrachat).
    Falls back to wiki if the target dataset is unavailable.
    """
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    ds_id, cfg, split, field = _load_dataset_for_domain(domain)
    kw: dict[str, object] = {"split": split, "streaming": True}
    if cfg is not None:
        kw["name"] = cfg
    dataset = load_dataset(ds_id, **kw)  # type: ignore[call-overload]
    samples = []
    for raw in dataset:
        text = _extract_text_from_sample(raw, field)
        if not text:
            continue
        tokens = tokenizer(
            text, max_length=max_length, truncation=True, return_tensors="pt"
        )
        if tokens["input_ids"].shape[1] >= min_length:
            samples.append(tokens)
        if len(samples) >= n_samples:
            break
    return samples
def build_global_token_set(
    model: AutoModelForCausalLM,
    calibration_samples: list[dict[str, torch.Tensor]],
    K: int = 64,
) -> list[int]:
    """Build global token set from top-K logits across all samples."""
    token_counter: Counter[int] = Counter()
    model.eval()  # type: ignore[attr-defined]
    device = next(model.parameters()).device  # type: ignore[attr-defined]
    with torch.no_grad():
        for sample in calibration_samples:
            inputs = {
                "input_ids": sample["input_ids"].to(device),
                "attention_mask": sample["attention_mask"].to(device),
            }
            outputs = model(**inputs)  # type: ignore[operator]
            logits = outputs.logits[:, -1, :]
            top_idx = logits.topk(K, dim=-1).indices[0]
            token_counter.update(top_idx.tolist())
    U = [tid for tid, _ in token_counter.most_common(K)]
    return sorted(U)
def load_models(
    large_name: str, small_name: str, dtype: torch.dtype = torch.float16
) -> tuple[torch.nn.Module, torch.nn.Module]:
    """Load large and small models with eager attention."""
    kwargs = {
        "torch_dtype": dtype,
        "attn_implementation": "eager",
    }
    large = AutoModelForCausalLM.from_pretrained(large_name, **kwargs)
    large.eval()  # type: ignore[attr-defined]
    small = AutoModelForCausalLM.from_pretrained(small_name, **kwargs)
    small.eval()  # type: ignore[attr-defined]
    return large, small
def extract_embeddings(
    model: AutoModelForCausalLM,
) -> torch.Tensor:
    """Extract embedding matrix from model."""
    return model.model.embed_tokens.weight.detach()  # type: ignore[attr-defined]


In [ ]:
import json
import os
import time
import torch
from transformers import AutoModelForCausalLM  # type: ignore[import-not-found]
def _layer_order(pairs: list[str], device: str) -> dict[str, list[str]]:
    late_to_early = sorted(pairs, key=lambda p: int(p.split("_")[1]), reverse=True)
    if device == "cpu":
        # CPU subset: only late→early, layers [23, 15, 7] or closest available
        wanted = [23, 15, 7]
        subset = [p for p in late_to_early if int(p.split("_")[1]) in wanted]
        if not subset:
            subset = late_to_early[:7]
        return {"late_to_early": subset}
    return {"late_to_early": late_to_early, "early_to_late": list(reversed(late_to_early))}
def main_17loradistill(
    small_model: str = "Qwen/Qwen3.5-0.8B",
    device: str = "auto",
    max_length: int = 64,
    min_length: int = 32,
    targets_dir: str = "results/targets",
    results_dir: str = "results",
):
    torch.manual_seed(42)
    start = time.time()
    if device == "auto":
        device = "cuda" if torch.cuda.is_available() else "cpu"
    if device == "cpu":
        torch.set_num_threads(4)
    print(f"Device: {device}")
    os.makedirs(results_dir, exist_ok=True)
    # -- Load small model -------------------------------------------------
    print(f"Loading small model: {small_model}")
    mS_template = small_model  # for re-loading fresh copies
    n_S = 24  # Qwen3.5-0.8B has 24 layers
    # -- Calibration data -------------------------------------------------
    n_samples = 2 if device == "cpu" else 4
    print(f"Loading {n_samples} calibration samples...")
    samples = load_calibration_data(
        small_model, max_length=max_length, min_length=min_length, n_samples=n_samples,
    )
    # -- Load target cache ------------------------------------------------
    cache_files = sorted(
        f for f in os.listdir(targets_dir)
        if f.endswith(".pt") and not f.startswith(".")
    )
    if not cache_files:
        raise FileNotFoundError(f"No target cache in {targets_dir}. Run Stage 0 first.")
    cache_by_pair: dict[str, list[dict]] = {}
    for cf in cache_files:
        data = torch.load(os.path.join(targets_dir, cf), weights_only=True)
        pair = data["pair"]
        cache_by_pair.setdefault(pair, []).append(data)
    all_pairs = sorted(cache_by_pair.keys(), key=lambda p: int(p.split("_")[1]))
    orders = _layer_order(all_pairs, device)
    steps = 100 if device == "cuda" else 40
    r = 4
    lr = 1e-3
    kl_weight = 0.1
    per_order_results = {}
    for order_name, pair_seq in orders.items():
        print(f"\n{'='*60}")
        print(f"Order: {order_name} ({len(pair_seq)} pairs)")
        print(f"{'='*60}")
        # Load fresh model for this order
        mS = AutoModelForCausalLM.from_pretrained(
            small_model, torch_dtype=torch.float32, attn_implementation="eager",
            low_cpu_mem_usage=True,
        ).to(device)
        mS.eval()
        # Cache original logits (updated after each layer)
        orig_logits_cache: list[torch.Tensor] = []
        with torch.no_grad():
            for s in samples[:n_samples]:
                ids = s["input_ids"].to(device)
                attn = s["attention_mask"].to(device)
                orig_logits_cache.append(
                    mS(input_ids=ids, attention_mask=attn).logits[:, -1, :]
                )
        per_layer_results = []
        for pair_key in pair_seq:
            targets = cache_by_pair[pair_key]
            l_L_str, l_S_str = pair_key.split("_")
            l_S = int(l_S_str)
            token_set = targets[0]["token_set"]
            print(f"\n  Pair {pair_key} (l_S={l_S})")
            # Freeze all params
            for p in mS.parameters():
                p.requires_grad = False
            # LoRA on lm_head: ΔW = (B @ A).T, A: [D,r], B: [r,V]
            V, D = mS.lm_head.weight.shape
            A_lh = torch.nn.Parameter(torch.randn(D, r, device=device) * 0.02)
            B_lh = torch.nn.Parameter(torch.zeros(r, V, device=device))
            # LoRA on target layer's attention output projection
            layer_mod = mS.model.layers[l_S]
            attn_mod = layer_mod.self_attn if hasattr(layer_mod, "self_attn") else layer_mod.linear_attn
            # Find o_proj weight
            o_proj_name = None
            for name, _ in attn_mod.named_parameters():
                if "o_proj" in name:
                    o_proj_name = name
                    break
            A_attn = None
            B_attn = None
            if o_proj_name:
                o_proj_weight = getattr(attn_mod, o_proj_name)
                V_l, D_l = o_proj_weight.shape
                A_attn = torch.nn.Parameter(torch.randn(D_l, r, device=device) * 0.02)
                B_attn = torch.nn.Parameter(torch.zeros(r, V_l, device=device))
            trainable = [A_lh, B_lh]
            if A_attn is not None:
                trainable.extend([A_attn, B_attn])
            for p in trainable:
                p.requires_grad = True
            optimizer = torch.optim.Adam(trainable, lr=lr)
            for step in range(steps):
                optimizer.zero_grad()
                total_loss = 0.0
                for si in range(min(n_samples, len(targets))):
                    s = samples[si]
                    ids = s["input_ids"].to(device)
                    attn = s["attention_mask"].to(device)
                    tgt = targets[si]
                    J_tgt_p = tgt["J_tgt_p"].to(device)
                    # Apply LoRA to lm_head for Jacobian computation
                    lm_orig = mS.lm_head.weight.data.clone()
                    delta_lm = (B_lh @ A_lh).T
                    mS.lm_head.weight.data = lm_orig + delta_lm
                    # Also apply LoRA to attention o_proj
                    if A_attn is not None and o_proj_name:
                        attn_orig = getattr(attn_mod, o_proj_name).clone()
                        delta_attn = B_attn @ A_attn
                        setattr(attn_mod, o_proj_name, attn_orig + delta_attn)
                    try:
                        J_S = compute_jacobian_for_tokens(mS, l_S, ids, attn, token_set)
                        p_L = tgt["p"].to(device)
                        J_S_p = softmax_jacobian(J_S, p_L)
                        cos_sim = torch.nn.functional.cosine_similarity(
                            J_S_p.float(), J_tgt_p.float(), dim=1
                        )
                        bridge_loss = (1.0 - cos_sim).mean()
                        # KL leash: need forward with LoRA applied
                        with torch.no_grad():
                            p_orig = torch.softmax(orig_logits_cache[si].float(), dim=-1)
                        logits_ed = mS(input_ids=ids, attention_mask=attn).logits[:, -1, :]
                        p_ed = torch.softmax(logits_ed.float(), dim=-1)
                        kl_loss = (p_ed * torch.log(p_ed / (p_orig + 1e-12))).mean()
                        total_loss += bridge_loss + kl_weight * kl_loss
                    except Exception as e:
                        print(f"    Step {step} sample {si} error: {e}")
                    finally:
                        # Restore weights
                        mS.lm_head.weight.data.copy_(lm_orig)
                        if A_attn is not None and o_proj_name:
                            setattr(attn_mod, o_proj_name, attn_orig)
                total_loss.backward()
                optimizer.step()
                if (step + 1) % 10 == 0 or step == 0:
                    print(f"    step {step+1}/{steps}: loss={total_loss.item():.4f}")
            # Freeze: apply LoRA permanently to lm_head
            delta_lm_final = (B_lh @ A_lh).T
            mS.lm_head.weight.data += delta_lm_final
            # Evaluate bridge
            bridge_vals = []
            for si in range(min(n_samples, len(targets))):
                s = samples[si]
                ids = s["input_ids"].to(device)
                attn = s["attention_mask"].to(device)
                tgt = targets[si]
                J_tgt_p = tgt["J_tgt_p"].to(device)
                J_S_ed = compute_jacobian_for_tokens(mS, l_S, ids, attn, token_set)
                p_L = tgt["p"].to(device)
                J_S_ed_p = softmax_jacobian(J_S_ed, p_L)
                cos = torch.nn.functional.cosine_similarity(
                    J_S_ed_p.float(), J_tgt_p.float(), dim=1
                ).mean().item()
                bridge_vals.append(cos)
            # KL to original
            kl_sum = 0.0
            for si in range(min(n_samples, len(orig_logits_cache))):
                s = samples[si]
                ids = s["input_ids"].to(device)
                attn_s = s["attention_mask"].to(device)
                logits_ed = mS(input_ids=ids, attention_mask=attn_s).logits[:, -1, :]
                p_ed = torch.softmax(logits_ed.float(), dim=-1)
                p_orig = torch.softmax(orig_logits_cache[si].float(), dim=-1)
                kl = (p_ed * torch.log(p_ed / (p_orig + 1e-12))).mean().item()
                kl_sum += kl
            kl_avg = kl_sum / min(n_samples, len(orig_logits_cache))
            layer_result = {
                "pair": pair_key,
                "bridge_mean": sum(bridge_vals) / len(bridge_vals) if bridge_vals else 0,
                "KL_ed_orig": kl_avg,
            }
            per_layer_results.append(layer_result)
            print(f"    bridge={layer_result['bridge_mean']:.4f}, KL={kl_avg:.4f}")
            # Update orig logits for next layer (cumulative edits)
            with torch.no_grad():
                for si, s in enumerate(samples[:n_samples]):
                    ids = s["input_ids"].to(device)
                    attn_s = s["attention_mask"].to(device)
                    orig_logits_cache[si] = mS(input_ids=ids, attention_mask=attn_s).logits[:, -1, :]
            # Clean up
            del A_lh, B_lh
            if A_attn is not None:
                del A_attn, B_attn
        per_order_results[order_name] = {"pairs": pair_seq, "per_layer": per_layer_results}
    # -- Save --------------------------------------------------------------
    result = {
        "per_order": per_order_results,
        "r": r,
        "steps": steps,
        "device": device,
        "elapsed_seconds": time.time() - start,
    }
    path = os.path.join(results_dir, "exp17_lora_distill.json")
    with open(path, "w") as f:
        json.dump(result, f, indent=2)
    print(f"\nSaved to {path}")
    print(f"Total time: {time.time() - start:.0f}s")


In [ ]:
# ---------- run exp17_lora_distill ----------
import inspect
sig = inspect.signature(main_17loradistill)
kwargs = {}
params = list(sig.parameters)
if 'small_model' in params: kwargs['small_model'] = SMALL
if 'large_model' in params: kwargs['large_model'] = LARGE
if 'model' in params: kwargs['model'] = SMALL
if 'device_L' in params: kwargs['device_L'] = device_L
if 'device_S' in params: kwargs['device_S'] = device_S
if 'device' in params: kwargs['device'] = device_S
if 'results_dir' in params: kwargs['results_dir'] = RES
if 'targets_dir' in params: kwargs['targets_dir'] = os.path.join(RES, 'targets')
# Remove conflicting keys: if both model and small_model, prefer specific
if 'small_model' in kwargs and 'model' in kwargs: del kwargs['model']
t0 = time.time()
try:
    main_17loradistill(**kwargs)
    elapsed = time.time() - t0
    print(f"[OK] exp17_lora_distill {elapsed:.0f}s")
except Exception as e:
    print(f"[FAIL] exp17_lora_distill: {e}")
    traceback.print_exc()
print("\nResults:")
if os.path.isdir(RES):
    try:
        for fn in sorted(os.listdir(RES)):
            print(f"  {fn} {os.path.getsize(os.path.join(RES, fn)):,} bytes")
    except OSError:
        pass
